In [102]:
from pathlib import Path
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.model_selection import train_test_split
from peft import LoraConfig, TaskType
from torch.utils.data import Dataset

random_state = 2026

In [110]:
def generate_dataset(path="./diabetes_binary_5050split_health_indicators_BRFSS2015.csv"):
  df = pd.read_csv(path)
  rename_cols = {
      "Diabetes_binary": "diabetes",
      "MentHlth": "days_mentally_ill",
      "Smoker": "smoking",
      "HvyAlcoholConsump": "alchohol_consumption",
      "PhysActivity": "physical_activity",
      "Fruits": "consumed_fruits",
      "Veggies": "consumed_vegetables",
      "PhysHlth": "days_physically_ill",
      "DiffWalk": "difficulty_walking",
      "Sex": "sex"
  }
  columns_filter = [col for col in rename_cols.keys()]
  df = df[columns_filter]
  df = df.rename(columns=rename_cols)

  df = df.dropna()
  return df



In [111]:
df = generate_dataset()
df

,diabetes,days_mentally_ill,smoking,alchohol_consumption,physical_activity,consumed_fruits,consumed_vegetables,days_physically_ill,difficulty_walking,sex
0,0.0,5.0,0.0,0.0,1.0,0.0,1.0,30.0,0.0,1.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,1.0,1.0,1.0,10.0,0.0,1.0
3,0.0,0.0,1.0,0.0,1.0,1.0,1.0,3.0,0.0,1.0
4,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
23404,0.0,7.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0
23405,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
23406,0.0,30.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
23407,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


In [6]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
model = AutoModelForSequenceClassification.from_pretrained("dmis-lab/biobert-base-cased-v1.1", num_labels=2, dtype=torch.bfloat16)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

In [7]:
tokenizer.pad_token = tokenizer.eos_token
model.to("cuda")

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [109]:
temp_df = df.head(50)

y = temp_df["diabetes"]
X = temp_df.drop(columns=["diabetes"])

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=random_state, shuffle=False)